# 1. Install JAX FDM

In [ ]:
!pip install -q -U jax-fdm compas_notebook
import jax_fdm
jax_fdm.__version__

# 2. Import libraries

In [ ]:
# compas
from compas.datastructures import Mesh
from compas.datastructures import Network
from compas.geometry import Line

# jax fdm
from jax_fdm.datastructures import FDNetwork

from jax_fdm.equilibrium import fdm
from jax_fdm.equilibrium import constrained_fdm

from jax_fdm.optimization import LBFGSBS
from jax_fdm.optimization import OptimizationRecorder

from jax_fdm.parameters import EdgeForceDensityParameter

from jax_fdm.goals import NodePointGoal

from jax_fdm.losses import RootMeanSquaredError
from jax_fdm.losses import Loss

from jax_fdm.visualization import LossPlotter
from jax_fdm.visualization import NotebookViewer

# 3. Download files from Github

In [ ]:
# Download files
!wget -P ./ https://raw.githubusercontent.com/arpastrana/jax_fdm/main/data/json/vault.json
!wget -P ./ https://raw.githubusercontent.com/arpastrana/jax_fdm/main/data/json/vault0.json

# 4. Load target surface

In [ ]:
# Import target network
network_target = Network.from_json("vault.json")

# Visualize target surface
viewer = NotebookViewer()
viewer.add(network_target, show_points=True, show_lines=False)
viewer.show()

# 5. Load initial pattern

In [ ]:
# Import JAX FDM network
network = FDNetwork.from_json("vault0.json")

# Visualize initial pattern
viewer = NotebookViewer()
viewer.add(network)
viewer.show()

# 6. Standard form-finding

In [ ]:
# Define equilibrium model on the network
supports = [node for node in network.nodes() if network.is_leaf(node)]
network.nodes_supports(supports)
network.nodes_loads([0.0, 0.0, -0.2])
network.edges_forcedensities(q=-1.0)

# Form-find the network
network_eq = fdm(network)

# Display form-found network
viewer = NotebookViewer()
viewer.add(network_eq, loadscale=3.0)
viewer.show()

# 7. Solve best-fit problem via gradient-based optimization

In [ ]:
# Define optimization parameters
parameters = []
for edge in network.edges():
    parameter = EdgeForceDensityParameter(edge, bound_low=-20.0, bound_up=0.0)
    parameters.append(parameter)

# Define best-fit goals
goals_bestfit = []
for node in network.nodes_free():
    target_xyz = network_target.node_coordinates(node)
    goal = NodePointGoal(node, target_xyz)
    goals_bestfit.append(goal)

# Set up objective function
error = RootMeanSquaredError(goals_bestfit)
loss = Loss(error)

# Solve constrained form-finding problem
optimizer = LBFGSBS()
recorder = OptimizationRecorder(optimizer)
network_c = constrained_fdm(network,
                            optimizer=optimizer,
                            loss=loss,
                            parameters=parameters,
                            maxiter=1000,
                            tol=1e-6,
                            callback=recorder)

In [ ]:
plotter = LossPlotter(loss, network)
plotter.plot(recorder.history)
plotter.show()

# 8. Visualize final results

In [ ]:
# Display solution
viewer = NotebookViewer()

# optimized network
viewer.add(network_c,
           edgewidth=(0.1, 0.3),
           edgecolor="fd",
           show_loads=False,
           loadscale=5.0)

# reference network
viewer.add(network_target.copy(cls=Network),
           show_points=True,
           show_lines=False)

# draw lines to target
for node in network_c.nodes():
    pt = network_c.node_coordinates(node)
    line = Line(pt, network_target.node_coordinates(node))
    viewer.add(line)

# show le crème
viewer.show()